# Self-Prior Multimodal Restoration Visualization

Display images from the `multimodal_restoration_*` folder as a tile.

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams["mathtext.fontset"] = "custom"
mpl.rcParams["mathtext.rm"] = "DejaVu Serif"
mpl.rcParams["mathtext.bf"] = "DejaVu Serif:bold"
mpl.rcParams["mathtext.it"] = "DejaVu Serif:italic"
mpl.rcParams["mathtext.default"] = "regular"


# BASE_DIR = ".../log/robot_mirror/rep5easy/eval_period500/nodecay/label03/20-03/multimodal_restoration_5500"
BASE_DIR = ".../log/robot_mirror/rep5easy/eval_period500/nodecay/2-0/multimodal_restoration_50000"
BASE_ROW_PATTERNS = [
    "input_vision_{:02d}.png",
    "vision_recon_v_{:02d}.png",
    "vision_recon_self_v_{:02d}.png",
    # "vision_recon_p_{:02d}.png",
    "vision_recon_self_p_{:02d}.png",
    # "render_from_proprio_v_{:02d}.png",
    # "render_from_proprio_self_v_{:02d}.png",
]
N_COLS = 5
BASE_ROW_LABELS = [
    "Right eye\nvision",
    "Reconst.\nfrom vision",
    "Reconst.\nfrom vision\n" + r"$\mathbf{with~self\text{-}prior}$",
    "Reconst.\n" + r"$\mathbf{from~proprio.}$" + "\n" + r"$\mathbf{with~self\text{-}prior}$",
]
DIFF_ROW_LABELS = [
    # "abs(vision_recon_v - input)",
    # "abs(vision_recon_self_v - input)",
    # "abs(vision_recon_p - input)",
    # "abs(vision_recon_self_p - input)",
    # "abs(render_from_proprio_v - input)",
    # "abs(render_from_proprio_self_v - input)",
]
ROW_LABELS = BASE_ROW_LABELS + DIFF_ROW_LABELS

# Row label font size (pt)
ROW_LABEL_FONTSIZE = 48

In [ ]:
def load_row(base_dir, pattern, n_cols):
    """Load one image row and return an array with shape (n_cols, H, W, C)."""
    imgs = []
    for i in range(n_cols):
        path = os.path.join(base_dir, pattern.format(i))
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Not found: {path}")
        img = np.array(Image.open(path))
        imgs.append(img)
    return np.array(imgs)


def build_rows(base_dir, row_patterns, n_cols):
    """Return row-image arrays for each given pattern."""
    return [load_row(base_dir, pattern, n_cols) for pattern in row_patterns]


def build_abs_diff_row(row_imgs, input_imgs):
    """Return pixel-wise absolute difference from input image as uint8."""
    diff = np.abs(row_imgs.astype(np.int16) - input_imgs.astype(np.int16))
    return diff.astype(np.uint8)


def build_tile(base_dir, row_patterns, n_cols):
    """Build a tile array from base rows plus optional absolute-difference rows."""
    base_rows = build_rows(base_dir, row_patterns, n_cols)
    input_row = base_rows[0]
    # diff_rows = [build_abs_diff_row(row_imgs, input_row) for row_imgs in base_rows[1:]]
    diff_rows = []
    all_rows = base_rows + diff_rows

    row_tiles = [np.concatenate(row_imgs, axis=1) for row_imgs in all_rows]
    tile = np.concatenate(row_tiles, axis=0)
    return tile

In [ ]:
tile = build_tile(BASE_DIR, BASE_ROW_PATTERNS, N_COLS)
print(f"Tile shape: {tile.shape}")

In [ ]:
# Number of image rows actually used
# (build_tile currently uses only base rows because diff_rows=[])
n_img_rows = len(BASE_ROW_PATTERNS)
n_label_rows = min(len(ROW_LABELS), n_img_rows)
tile_h, tile_w = tile.shape[:2]

# Paper-friendly serif setup (fallback: Times / Liberation Serif / DejaVu Serif)
_rc = {
    "font.family": "serif",
    "font.serif": [
        "Times New Roman",
        "Times",
        "Nimbus Roman",
        "Liberation Serif",
        "DejaVu Serif",
    ],
    "axes.unicode_minus": False,
    "savefig.pad_inches": 0,
}

with plt.rc_context(_rc):
    if ROW_LABELS and n_img_rows > 0:
        _wspace = 0.02
        _fig_w = 20
        _img_col_share = N_COLS / (1 + N_COLS) / (1 + _wspace / 2)
        _fig_h = _fig_w * _img_col_share * tile_h / tile_w

        fig = plt.figure(figsize=(_fig_w, _fig_h))
        gs = fig.add_gridspec(
            1,
            2,
            width_ratios=[1, N_COLS],
            left=0,
            right=1,
            bottom=0,
            top=1,
            wspace=_wspace,
        )
        ax_label = fig.add_subplot(gs[0, 0])
        ax_img = fig.add_subplot(gs[0, 1])

        # Keep original aspect ratio and explicitly map data coordinates
        # so label y positions align exactly.
        ax_img.imshow(tile, aspect="equal", extent=(0, tile_w, tile_h, 0))
        ax_img.set_xlim(0, tile_w)
        ax_img.set_ylim(tile_h, 0)
        ax_img.set_axis_off()

        ax_label.sharey(ax_img)
        ax_label.set_xlim(0, 1)
        ax_label.set_axis_off()

        # With equal aspect and axis box constraints, uniform data-y splits
        # can drift from actual rendered pixel rows.
        # After draw(), use the image bbox to map row centers back to data-y.
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        im = ax_img.get_images()[0]
        bbox = im.get_window_extent(renderer)
        for i, label in enumerate(ROW_LABELS[:n_label_rows]):
            y_disp = bbox.y1 - (i + 0.5) / n_label_rows * bbox.height
            x_disp = bbox.x0 + 0.5 * bbox.width
            _, y_data = ax_label.transData.inverted().transform((x_disp, y_disp))
            ax_label.text(
                1.0,
                y_data,
                label,
                ha="right",
                va="center",
                transform=ax_label.transData,
                fontsize=ROW_LABEL_FONTSIZE,
            )
    else:
        fig, ax_img = plt.subplots(1, 1, figsize=(20, 18))
        ax_img.imshow(tile)
        ax_img.set_axis_off()

    plt.show()

In [ ]:
# (Optional) Save the full figure with labels as PDF (keep vector text).
out_path = os.path.join(os.path.dirname(BASE_DIR), "multimodal_restoration_tile.pdf")
fig.savefig(out_path, format="pdf", bbox_inches="tight", pad_inches=0)
print(f"Saved: {out_path}")